# Fine-Tuning Example for SMAP Recharge Flux PINN

This notebook demonstrates how to fine-tune a pretrained PINN model on new boundary condition data.

**Use case**: Train on Jan-Mar data, then fine-tune on May-Aug data with different rainfall patterns.

## Setup

In [ ]:
import sys
import os

# Add parent directory to path to import from src/
notebook_dir = os.path.dirname(os.path.abspath("__file__"))
project_root = os.path.dirname(notebook_dir)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

import torch
import numpy as np
import matplotlib.pyplot as plt

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")

## Configuration

In [ ]:
# Path to base checkpoint (from initial training on Jan-Mar data)
BASE_CHECKPOINT = 'checkpoints/checkpoint_final.pt'

# Network configurations (must match base training)
h_net_config = {'hidden_dim': 64, 'num_layers': 4}
zb_net_config = {'hidden_dim': 32, 'num_layers': 3}

## Load New Boundary Condition Data

**MODIFY THIS SECTION** to use your actual May-Aug data.

For now, we generate synthetic data as an example.

In [ ]:
from src.surf_flux import synth_surface_flux

# === OPTION 1: Synthetic data (for testing) ===
t_new, q_new = synth_surface_flux(
    total_days=15,        # Same duration as base training
    dt_minutes=30,
    storm_rate_per_day=0.25,  # Different rainfall pattern
    min_storm_hours=4,
    seed=42
)
new_q0_data = (t_new.tolist(), q_new.tolist())

# === OPTION 2: Load from .mat file (YOUR ACTUAL DATA) ===
# Uncomment and modify this to use your real data:
# from scipy.io import loadmat
# mat = loadmat("../data/may_aug_q0_data.mat")
# q0_data_may_aug = -mat["q0_data"]
# q0_data_may_aug[0] = 0
# T_may_aug = np.arange(len(q0_data_may_aug)) * 1800  # 30 min intervals
# new_q0_data = (T_may_aug.tolist(), q0_data_may_aug.tolist())

print(f"New data: {len(new_q0_data[0])} points over {new_q0_data[0][-1]/86400:.1f} days")
print(f"Flux range: {min(new_q0_data[1])*1e6:.2f} to {max(new_q0_data[1])*1e6:.2f} μm/s")

### Visualize New Data

In [ ]:
# Plot the new surface flux
plt.figure(figsize=(10, 4))
plt.plot(np.array(new_q0_data[0])/86400, np.array(new_q0_data[1])*1e6, '-', lw=1, label='New flux (May-Aug)')
plt.axhline(0, color='k', lw=0.8, alpha=0.5)
plt.xlabel("Time [days]")
plt.ylabel("Surface flux q [μm/s]")
plt.title("New Surface Flux Data")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## Detect Spike Events

In [ ]:
from src.spike_detection import detect_spike_events

q_tensor = torch.tensor(new_q0_data[1])
spike_events, _ = detect_spike_events(
    q_tensor,
    threshold_method='std',
    threshold_value=1.0,
    expansion_window=5,
    merge_distance=3,
    min_event_size=2,
    device=device
)

print(f"Detected {len(spike_events)} spike events in new data")

# Visualize spikes
plt.figure(figsize=(12, 4))
plt.plot(q_tensor.cpu().numpy(), label='New flux', alpha=0.7)
for i, event_indices in enumerate(spike_events):
    plt.scatter(event_indices.cpu().numpy(), q_tensor[event_indices].cpu().numpy(), 
                label=f'Spike {i+1}', s=40)
plt.xlabel('Index')
plt.ylabel('Flux')
plt.title('Detected Spike Events in New Data')
plt.legend()
plt.tight_layout()
plt.show()

## Set Initial Water Table Depth

**MODIFY THIS**: Estimate from April observations or end of March simulation.

In [ ]:
# Initial water table depth for May (estimate from observations or previous model)
zb_initial = 1.8  # meters
print(f"Initial water table depth: {zb_initial} m")

## Fine-Tune Model

In [ ]:
from src.train_loop import finetune_pinn

print("="*70)
print("Starting Fine-Tuning")
print("="*70)

model, losses, comps, sample_losses, sample_comps, sample_epochs = finetune_pinn(
    checkpoint_path=BASE_CHECKPOINT,
    new_q0_data=new_q0_data,
    zb_initial=zb_initial,
    h_net_config=h_net_config,
    zb_net_config=zb_net_config,

    # Fine-tuning hyperparameters (lower lr, fewer epochs than base)
    n_epochs=10000,
    learning_rate=1e-4,

    # Training parameters (same as base)
    cache_size=10000,
    batch_size=500,
    resample_freq=500,
    boundary_ratio=0.7,
    high_residual_ratio=0.3,
    temperature=1.0,
    n_events=6,
    batch_size_bc=120,

    # Fixed weights (no adaptive learning)
    weight_update_freq=1e10,
    weight_lr=0.1,
    use_initial_scales=True,

    # Device and GPU settings
    device=device,
    use_multi_gpu=True,
    use_amp=False,  # Set True for memory savings on GPU
    grad_accumulation_steps=1,

    # Checkpointing (isolated from base training)
    checkpoint_dir='checkpoints_finetune',
    checkpoint_freq=2000,  # Save every 2000 epochs
    keep_last_n_checkpoints=3,

    # Spike events
    spike_events=spike_events,
)

print("\n" + "="*70)
print("Fine-Tuning Complete!")
print("="*70)

## Visualize Results

In [ ]:
from src.visualization import plot_comprehensive_results

# Plot comprehensive results
plot_comprehensive_results(model, new_q0_data, device=device)

In [ ]:
from src.visualization import plot_training_losses

# Plot training losses
plot_training_losses(losses, comps, sample_losses, sample_comps, sample_epochs)

## Summary

Fine-tuning complete! Your model is saved at:
- `checkpoints_finetune/finetune_final.pt`

### Next Steps:
1. Validate against May-Aug observations
2. Use fine-tuned model for prediction
3. Compare with base model performance